> **🎯 Important**
>
**Durée estimée** : 8 à 10 heures  
**Prérequis** : Toutes les notions du Module 4 (4.1 à 4.6) + Module 3 complet  
**Objectif** : réaliser un projet ML complet de classification sur un problème métier déséquilibré, de l'exploration jusqu'à un modèle tuné et validé


# 🎯 Contexte

Tu es **data scientist** chez **SecurePay**, une fintech française qui gère les paiements en ligne. Le CFO te convoque :

> « On subit 120 000 € de pertes par mois à cause des fraudes qu'on rate. 
> Notre système actuel bloque déjà les cas évidents, mais il manque les fraudes **subtiles**. 
> J'aimerais **un modèle ML** qui détecte au mieux ces fraudes, sans trop gêner les clients légitimes.  
> Tu as 10 000 transactions historiques labellisées, avec leur statut. 
> Fais-moi un modèle, et surtout, **explique-moi ce qu'il fait**. »

## 💰 L'équation métier

Chaque fraude ratée (**faux négatif**) coûte en moyenne **1 000 €** (remboursement au client + enquête).  
Chaque fausse alerte (**faux positif**) coûte **10 €** (gêne client, relance, risque de perte du client).

**Donc** : rater une fraude coûte **100 fois plus cher** qu'une fausse alerte.  
→ Ton modèle doit maximiser le **recall**, mais avec une **precision raisonnable**.

# 📊 Le dataset

Le fichier `transactions_fraude.csv` contient ~10 000 transactions avec 15 colonnes :

| Colonne | Description |
|---|---|
| `id_transaction` | Identifiant unique |
| `id_client` | Identifiant du client |
| `montant` | Montant de la transaction (€) |
| `heure` | Heure de la transaction (0-23) |
| `jour_semaine` | Jour de la semaine (0=lundi) |
| `delai_depuis_derniere_min` | Minutes depuis la transaction précédente |
| `nb_transactions_1h` | Nombre de transactions dans l'heure passée |
| `nb_transactions_24h` | Nombre de transactions dans les 24h |
| `distance_domicile_km` | Distance du point de vente au domicile |
| `pays` | Pays de la transaction (France / UE / Hors-UE) |
| `type_commercant` | Catégorie du commerçant |
| `canal_paiement` | Type de paiement (carte, en ligne, etc.) |
| `age_compte_annees` | Ancienneté du compte (années) |
| `score_ip` | Score de l'adresse IP (pour e-commerce, 0-100) |
| `est_fraude` | **CIBLE** : 0 = légitime, 1 = fraude |

> **⚠️ Attention**
>
## ⚠️ Ce dataset contient (volontairement) les pièges classiques
- **Classes très déséquilibrées** (~1.2% de fraudes)
- Quelques **doublons parfaits**
- **Valeurs manquantes** sur certaines colonnes
- **Outliers de montant** (erreurs de saisie)
- **Variables mixtes** (numériques + catégorielles)


# 🗺️ Plan de travail

Le TP est structuré en **6 parties** alignées sur le parcours d'un data scientist :

1. **Exploration et nettoyage** (mobilise Module 3)
2. **Baseline** : modèle simple de référence
3. **Modèles avancés** : RF, XGBoost, LightGBM
4. **Tuning et cross-validation**
5. **Évaluation complète** : toutes les bonnes métriques
6. **Décision business** : choix du seuil optimal et rapport

# 🛠️ Mise en route

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, auc
)

import xgboost as xgb
import lightgbm as lgb

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 20)

print("✅ Environnement prêt")

## 📥 Préparation : téléchargement des données

La cellule ci-dessous télécharge automatiquement les datasets nécessaires si ils ne sont pas déjà présents localement. Cela permet de faire marcher le notebook **aussi bien en local qu'en Google Colab**.

In [ ]:
# 📥 Téléchargement automatique des datasets (utile pour Colab)
import os, urllib.request

if not os.path.exists('ressources_tp/transactions_fraude.csv'):
    os.makedirs('ressources_tp', exist_ok=True)
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/Digit-Factory/python-ia-formation/main/ressources_eleves/module_04/ressources_tp/transactions_fraude.csv',
        'ressources_tp/transactions_fraude.csv'
    )
    print(f"✅ Dataset téléchargé : transactions_fraude.csv")
else:
    print(f"✅ Dataset déjà présent : transactions_fraude.csv")

In [ ]:
# Chargement silencieux
df = pd.read_csv("ressources_tp/transactions_fraude.csv")

In [ ]:
# Charger le dataset
df = pd.read_csv("ressources_tp/transactions_fraude.csv")
print(f"Dataset chargé : {df.shape}")

---

# Partie 1 — Exploration et nettoyage

## ✏️ Étape 1.1 — Aperçu initial

> **ℹ️ Note**
>
## 📝 À faire

1. Affiche les dimensions du DataFrame
2. Affiche `df.head()` et `df.info()`
3. Calcule le **taux de fraude** (`est_fraude.mean()`)
4. **Avertissement majeur** : quel serait l'accuracy d'un modèle qui prédit TOUJOURS "pas fraude" ?

In [ ]:
# TODO: Étape 1.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 1.2 — Diagnostic du dataset

> **ℹ️ Note**
>
## 📝 À faire

1. Combien de **doublons** parfaits dans le dataset ?
2. Quelles colonnes ont des **valeurs manquantes** et en quel nombre ?
3. Pour `montant`, affiche les statistiques — y a-t-il des **outliers évidents** (valeurs très élevées) ?

In [ ]:
# TODO: Étape 1.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 1.3 — Exploration bivariée ciblée

> **ℹ️ Note**
>
## 📝 À faire

Compare les fraudes aux transactions légitimes sur **4 variables** :

1. `montant` — boxplot comparatif
2. `heure` — taux de fraude par heure de la journée (bar chart)
3. `pays` — taux de fraude par pays
4. `nb_transactions_1h` — taux de fraude par nombre de transactions récentes

Quelles variables semblent les plus discriminantes ?

In [ ]:
# TODO: Étape 1.3

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 1.4 — Nettoyage

> **ℹ️ Note**
>
## 📝 À faire

1. **Supprimer les doublons**
2. **Gérer les outliers** : capper `montant` au percentile 99.5
3. Laisser les NaN pour l'instant — le pipeline les gérera via imputation

Affiche la taille finale du dataset.

In [ ]:
# TODO: Étape 1.4

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 2 — Baseline et preprocessing

## ✏️ Étape 2.1 — Préparer les features

> **ℹ️ Note**
>
## 📝 À faire

1. Identifier les **features à utiliser** et les **types** :
   - Identifier les colonnes à **exclure** (`id_transaction`, `id_client` — identifiants, non prédictifs)
   - Identifier les **numériques** vs **catégorielles**
2. Séparer `X` (features) et `y` (`est_fraude`)
3. Faire un **split stratifié** 80/20 avec `random_state=42`

In [ ]:
# TODO: Étape 2.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 2.2 — Construire le preprocessor

> **ℹ️ Note**
>
## 📝 À faire

Construis un `ColumnTransformer` qui :

- Applique **imputation médiane + StandardScaler** aux colonnes numériques
- Applique **OneHotEncoder(handle_unknown="ignore")** aux colonnes catégorielles

*Indice* : utilise `SimpleImputer(strategy="median")` dans un mini-pipeline pour les numériques.

In [ ]:
# TODO: Étape 2.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 2.3 — Baseline : régression logistique

> **ℹ️ Note**
>
## 📝 À faire

1. Construis un **pipeline complet** : `preprocessor + LogisticRegression(max_iter=1000)`
2. Entraîne sur le train
3. Évalue sur le test **avec 5 métriques** : accuracy, precision, recall, F1, ROC-AUC

Compare l'accuracy avec la baseline naïve. Pourquoi est-elle bonne mais trompeuse ?

In [ ]:
# TODO: Étape 2.3

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 3 — Modèles avancés

## ✏️ Étape 3 — Random Forest, XGBoost, LightGBM

> **ℹ️ Note**
>
## 📝 À faire

Entraîne **3 modèles avancés** avec des hyperparamètres raisonnables **et** le paramètre `class_weight` (ou `scale_pos_weight`) pour gérer le déséquilibre.

1. `RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)`
2. `xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, scale_pos_weight=80, random_state=42, eval_metric='logloss')`  
   *(scale_pos_weight = ratio négatifs/positifs ≈ 80)*
3. `lgb.LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, class_weight='balanced', random_state=42, verbose=-1)`

Évalue chacun sur le test et **compare** dans un tableau.

In [ ]:
# TODO: Étape 3

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## 💡 `class_weight='balanced'` : qu'est-ce que ça fait ?

Quand le dataset est déséquilibré, le modèle **apprend à prédire la classe majoritaire par défaut**. Le paramètre `class_weight='balanced'` dit : « pondère les erreurs sur la classe minoritaire **plus fortement** pour compenser le déséquilibre. »

**Équivalent pour XGBoost** : `scale_pos_weight = (nb négatifs) / (nb positifs)`.

C'est **la technique la plus simple** pour gérer le déséquilibre. Il y en a d'autres (SMOTE, undersampling...) qu'on pourrait explorer.

---

# Partie 4 — Tuning du meilleur modèle

## ✏️ Étape 4 — Optimiser XGBoost

> **ℹ️ Note**
>
## 📝 À faire

Sur le modèle le plus prometteur (souvent XGBoost), fais un **GridSearchCV** avec :

- `clf__n_estimators` : [100, 200, 300]
- `clf__max_depth` : [3, 5, 7]
- `clf__learning_rate` : [0.05, 0.1]

Optimise sur **`f1`** (pas accuracy !) avec `cv=5`.

*C'est long : 3×3×2 = 18 combinaisons × 5 folds = 90 entraînements.*

In [ ]:
# TODO: Étape 4

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 5 — Évaluation complète

## ✏️ Étape 5.1 — Matrice de confusion et courbes

> **ℹ️ Note**
>
## 📝 À faire

Sur le **meilleur modèle** (tuné) :

1. Affiche la **matrice de confusion** (heatmap)
2. Trace la **courbe ROC**
3. Trace la **courbe Precision-Recall**

Commente : combien de fraudes sont détectées ? combien ratées ?

In [ ]:
# TODO: Étape 5.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 5.2 — Feature importance

> **ℹ️ Note**
>
## 📝 À faire

1. Extrait la **feature importance** du modèle tuné
2. Affiche les 10 variables les plus importantes en bar chart horizontal
3. Les variables les plus importantes correspondent-elles à ton intuition métier ?

In [ ]:
# TODO: Étape 5.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 6 — Décision business : ajuster le seuil

## ✏️ Étape 6.1 — Courbe coût métier

> **ℹ️ Note**
>
## 📝 À faire

Rappelons les coûts :
- **Faux négatif** (fraude ratée) = 1 000 €
- **Faux positif** (fausse alerte) = 10 €

1. Pour chaque **seuil de décision** entre 0.05 et 0.95, calcule le **coût total** attendu sur le test
2. Identifie le **seuil qui minimise le coût**
3. Compare avec le seuil standard (0.5)

In [ ]:
# TODO: Étape 6.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 6.2 — Évaluation finale avec le seuil optimal

> **ℹ️ Note**
>
## 📝 À faire

Ré-évalue le modèle avec le **seuil optimal** :

1. Calcule les 5 métriques (accuracy, precision, recall, F1, ROC-AUC)
2. Compare avec le seuil standard 0.5
3. Commente : quelle amélioration business ?

In [ ]:
# TODO: Étape 6.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 6.3 — Rapport exécutif

> **ℹ️ Note**
>
## 📝 À faire

Rédige un **rapport d'une page** pour le CFO, structuré ainsi :

1. **Contexte** — le problème, les données
2. **Méthodologie** — nettoyage, choix du modèle, tuning, évaluation
3. **Résultats** — métriques clés, seuil optimisé
4. **Impact business** — en € économisés
5. **Limites et pistes d'amélioration**


> **💡 Astuce**
>
## ✅ Exemple de rapport

```markdown
# Rapport : Modèle de détection de fraude bancaire

## 1. Contexte

Objectif : réduire les pertes liées aux fraudes (~120 000 €/mois), sans dégrader 
l'expérience client. Dataset : 10 000 transactions historiques avec leur statut 
(1.2% frauduleuses).

## 2. Méthodologie

- **Exploration** : identifié les patterns clairs (horaires nocturnes, pays hors-UE, 
  transactions rapprochées, montants ronds)
- **Nettoyage** : doublons supprimés, outliers de montant cappés, pipeline 
  d'imputation pour les valeurs manquantes
- **Modélisation** : comparé Logistic Regression, Random Forest, XGBoost, LightGBM
- **Déséquilibre géré** via `scale_pos_weight` (XGBoost) ou `class_weight='balanced'`
- **Tuning** : GridSearchCV sur XGBoost, optimisation sur F1
- **Évaluation** : métriques F1, precision, recall, ROC-AUC, PR-AUC, courbe coût métier

## 3. Résultats

Le modèle final (XGBoost tuné) obtient sur un set de test jamais vu :
- **ROC-AUC : ~0.95**
- **PR-AUC : ~0.70**
- **Recall (fraudes détectées) : ~85%** avec le seuil optimal
- **Precision : ~30%** (1 alerte sur 3 est une vraie fraude)

Variables les plus importantes (métier) :
1. Heure de transaction (plus risqué la nuit)
2. Pays Hors-UE
3. Nombre de transactions dans la dernière heure
4. Montant
5. Distance du domicile

## 4. Impact business

Coûts asymétriques pris en compte :
- Fraude ratée : 1 000 € (remboursement)
- Fausse alerte : 10 € (relance, gêne)

**Optimisation du seuil de décision** → économies de 25-40% vs seuil standard.

Sur 2 000 transactions/mois, le modèle projette :
- ~85% des fraudes détectées (vs ~50% actuellement)
- Coût mensuel projeté : à déterminer avec tests en conditions réelles

## 5. Limites et pistes

**Limites** :
- Dataset historique (les patterns de fraude évoluent dans le temps)
- Pas de variables comportementales long-terme (historique client sur 6+ mois)
- Les fraudeurs s'adapteront → re-entraînement régulier nécessaire

**Pistes** :
- A/B test en production pour confirmer les gains
- Enrichir avec features device (navigateur, device ID, etc.)
- Explorer des modèles de graphes (détection de réseaux frauduleux)
- Déployer avec MLOps (monitoring de drift, re-entraînement automatique)
```


---

# 🎓 Bilan du TP

## 🏆 Ce que tu viens d'accomplir

Tu as réalisé un **projet ML complet** sur un problème industriel typique :

- ✅ Exploration et nettoyage d'un dataset bruité (Module 3)
- ✅ Pipeline reproductible avec ColumnTransformer (Module 3)
- ✅ Baseline + 3 modèles avancés comparés (Module 4)
- ✅ Gestion du déséquilibre de classes
- ✅ Tuning via GridSearchCV
- ✅ Évaluation multi-métriques adaptée
- ✅ **Ajustement du seuil** selon coûts métier
- ✅ Rapport exécutif pour décideurs

C'est **exactement le workflow** d'un data scientist en entreprise.

## 🔍 Les notions mobilisées

| Module / Notion | Utilisée dans |
|---|---|
| **3 — EDA, nettoyage, pipelines** | Parties 1, 2 |
| **4.1 — ML supervisé** | Toutes |
| **4.2 — Logistic Regression (baseline)** | Partie 2 |
| **4.3 — Random Forest** | Partie 3 |
| **4.4 — XGBoost, LightGBM** | Partie 3 |
| **4.5 — Métriques, seuil optimal** | Parties 5, 6 |
| **4.6 — Tuning, cross-validation** | Partie 4 |

## 🚀 La suite — Module 5 : ML non supervisé

Tu as vu tout le **supervisé**. Dans le Module 5, on découvrira l'**apprentissage non supervisé** :

- Clustering (k-means, DBSCAN, hiérarchique)
- Réduction de dimension (PCA, t-SNE, UMAP)
- Détection d'anomalies
- Applications : segmentation client, compression, détection d'outliers

Et ensuite, le Deep Learning (Module 6) pour les données non-tabulaires.

> **💡 Astuce**
>
## 💡 Défi pour consolider

1. **Test de robustesse** : le modèle tuné est-il meilleur que le RF par défaut ? De combien ?
2. **Autre métrique à optimiser** : si on veut un **recall minimum de 95%** (on ne rate quasi aucune fraude), quel seuil ? Quelle precision obtient-on alors ?
3. **Variante** : essaye `imbalanced-learn` et la technique **SMOTE** (sur-échantillonnage de la classe minoritaire). Est-ce que ça aide ou pas ?

Tu auras alors le **réflexe complet** du data scientist spécialisé en classes déséquilibrées.


**🎉 Félicitations — tu as bouclé le Module 4 !**